In [0]:
# DLT works with three types of datasets
# Streaming tables (Permenant/temporary) -- used as append daa sources, incremental data
# Materilaized views -- Used for transformations, aggregations, or computations
import dlt

In [0]:
#Ceate a streaming table for orders 
@dlt.table(
  table_properties ={"quality": "Bronze"},
  comment = "Orders bronze table"
  )
def orders_bronze():
    df = spark.readStream.table("learning.bronze.orders_raw")
    return df

In [0]:
#Ceate a Materialized View for customers 
@dlt.table(
  table_properties ={"quality": "Bronze"},
  comment = "customers bronze table",
  name = "customer_bronze"
  )
def cust_bronze():
    df = spark.read.table("learning.bronze.customer_raw")
    return df

In [0]:
#Create a view to Join orders with cutsomers
@dlt.view(
  comment = "Joined View"
)
def joined_vw():
    df_o = spark.read.table("LIVE.orders_bronze")
    df_c = spark.read.table("LIVE.customer_bronze")

    df_joined = df_o.join(df_c, how="left_outer", on = df_o.o_custkey == df_c.c_custkey)
    return df_joined 



In [0]:

from pyspark.sql.functions import current_timestamp, count
#Ceate a Materialized View for extra column 
@dlt.table(
  table_properties ={"quality": "Silver"},
  comment = "Joined Silver table",
  name = "joined_silver"
)
def joined_silver():
    df = spark.read.table("LIVE.joined_vw").withColumn("__Insert_date", current_timestamp())
    return df

In [0]:
from pyspark.sql.functions import count, current_timestamp

@dlt.table(
  table_properties = {"quality": "Gold"},
  comment = "Orders aggregated table",
  name = "orders_agg_gold"
)
def orders_agg_gold():
    df = spark.read.table("LIVE.joined_silver")
    df_final = (
        df.groupBy("c_mktsegment")
        .agg(
            count("o_orderkey").alias("Sum_orders")
        )
        .withColumn("__insert_date", current_timestamp())
    )
    return df_final